In [1]:
import os
import librosa
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, Dropout
from tensorflow.keras.utils import to_categorical

# ==========================================
# 1. CONFIGURATION
# ==========================================
CREMA_PATH = r"AudioWAV" 

emotion_map = {
    'ANG': 'angry', 'DIS': 'disgust', 'FEA': 'fear',
    'HAP': 'happy', 'NEU': 'neutral', 'SAD': 'sad'
}

X = []
y = []
MAX_TIME_FRAMES = 128

print("Extracting sequential MFCC matrices. Please wait...")

for root, dirs, files in os.walk(CREMA_PATH):
    for file in files:
        if file.endswith('.wav'):
            file_path = os.path.join(root, file)
            parts = file.split('_')
            
            if len(parts) >= 3:
                emotion_code = parts[2].upper()
                emotion = emotion_map.get(emotion_code, 'unknown')
                
                if emotion != 'unknown':
                    try:
                        audio_data, sample_rate = librosa.load(file_path, duration=3)
                        
                        mfccs = librosa.feature.mfcc(y=audio_data, sr=sample_rate, n_mfcc=40)
                        
                        if mfccs.shape[1] < MAX_TIME_FRAMES:
                            pad_width = MAX_TIME_FRAMES - mfccs.shape[1]
                            mfccs = np.pad(mfccs, ((0, 0), (0, pad_width)), mode='constant')
                        else:
                            mfccs = mfccs[:, :MAX_TIME_FRAMES]
                            
                        X.append(mfccs)
                        y.append(emotion)
                        
                    except Exception as e:
                        continue

X = np.array(X)
y = np.array(y)

print("\n--- Structural Extraction Verification ---")
print(f"Samples successfully processed: {len(X)}")
print(f"New Feature Matrix Shape (Samples, Features, Time): {X.shape}")

Extracting sequential MFCC matrices. Please wait...

--- Structural Extraction Verification ---
Samples successfully processed: 7442
New Feature Matrix Shape (Samples, Features, Time): (7442, 40, 128)


In [2]:
# ==========================================
# 3. DATA PREPARATION & SPLITTING
# ==========================================
label_encoder = LabelEncoder()
y_encoded = to_categorical(label_encoder.fit_transform(y))
np.save('crema_classes.npy', label_encoder.classes_)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

print(f"Train set input shape: {X_train.shape}")

Train set input shape: (5953, 40, 128)


In [3]:
# ==========================================
# 4. CNN MODEL ARCHITECTURE & TRAINING
# ==========================================

from tensorflow.keras.callbacks import ReduceLROnPlateau

model = Sequential([
    Conv1D(256, kernel_size=5, strides=1, padding='same', activation='relu', input_shape=(40, 128)),
    MaxPooling1D(pool_size=2, padding='same'),
    Dropout(0.3),
    
    Conv1D(128, kernel_size=5, strides=1, padding='same', activation='relu'),
    MaxPooling1D(pool_size=2, padding='same'),
    Dropout(0.3),
    
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(len(label_encoder.classes_), activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5, 
    patience=3, 
    min_lr=0.00001
)
print("Running 1D CNN training on timeline-preserved audio arrays...")
model.fit(
    X_train, 
    y_train, 
    epochs=50, 
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[reduce_lr]
)
model.summary()

Running 1D CNN training on timeline-preserved audio arrays...

C:\Users\kisha\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Epoch 1/50
187/187 ━━━━━━━━━━━━━━━━━━━━ 8s 33ms/step - accuracy: 0.1732 - loss: 8.7194 - val_accuracy: 0.1666 - val_loss: 1.7932 - learning_rate: 0.0010
Epoch 2/50
187/187 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.1824 - loss: 1.7924 - val_accuracy: 0.2357 - val_loss: 1.7762 - learning_rate: 0.0010
Epoch 3/50
187/187 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.2389 - loss: 1.7582 - val_accuracy: 0.2626 - val_loss: 1.7456 - learning_rate: 0.0010
Epoch 4/50
187/187 ━━━━━━━━━━━━━━━━━━━━ 10s 30ms/step - accuracy: 0.2432 - loss: 1.7456 - val_accuracy: 0.2686 - val_loss: 1.7238 - learning_rate: 0.0010
Epoch 5/50
187/187 ━━━━━━━━━━━━━━━━━━━━ 11s 35ms/step - accuracy: 0.2587 - loss: 1.7219 - val_accuracy: 0.2518 - val_loss: 1.7344 - learning_rate: 0.0010
Epoch 6/50
187/187 ━━━━━━━━━━━━━━━━━━━━ 11s 37ms/step - accuracy: 0.2758 - loss: 1.6983 - val_accuracy: 0.2270 - val_loss: 1.7398 - learning_rate: 0.0010
Epoch 7/50
187/187 ━━━━━━━━━━━━━━━━━━━━ 10s 36ms/step - accuracy: 0.2745 - l

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                      │ (None, 40, 256)             │         164,096 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d (MaxPooling1D)         │ (None, 20, 256)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 20, 256)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ (None, 20, 128)             │         163,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_1 (MaxPooling1D)       │ (None, 10, 128)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 10, 128)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         327,936 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 6)                   │           1,542 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,972,628 (7.52 MB)

 Trainable params: 657,542 (2.51 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,315,086 (5.02 MB)

In [ ]:
model.save('crema_emotion_model.h5')
print("Done! Model file exported and saved as 'crema_emotion_model.h5'")

Done! Model file exported and saved as 'crema_emotion_model.h5'


In [ ]:
from tensorflow.keras.models import load_model

# ==========================================
# 1. CONFIGURATION
# ==========================================
TEST_FILE_PATH = r"./AudioWAV/1001_ITH_DIS_XX.wav" 
MODEL_PATH = 'crema_emotion_model.h5'
CLASSES_PATH = 'crema_classes.npy'
MAX_TIME_FRAMES = 128

# ==========================================
# 2. FILE VALIDATION
# ==========================================
if not os.path.exists(TEST_FILE_PATH):
    print(f"❌ Error: Target test audio file not found at: {TEST_FILE_PATH}")
elif not os.path.exists(MODEL_PATH) or not os.path.exists(CLASSES_PATH):
    print("❌ Error: Missing trained model or classes file. Run your training cells first!")
else:
    model = load_model(MODEL_PATH)
    classes = np.load(CLASSES_PATH, allow_pickle=True)

    audio_data, sample_rate = librosa.load(TEST_FILE_PATH, duration=3)
    
    mfccs = librosa.feature.mfcc(y=audio_data, sr=sample_rate, n_mfcc=40)
    
    if mfccs.shape[1] < MAX_TIME_FRAMES:
        pad_width = MAX_TIME_FRAMES - mfccs.shape[1]
        mfccs = np.pad(mfccs, ((0, 0), (0, pad_width)), mode='constant')
    else:
        mfccs = mfccs[:, :MAX_TIME_FRAMES]
        
    features = np.expand_dims(mfccs, axis=0)

    predictions = model.predict(features, verbose=0)
    predicted_index = np.argmax(predictions)
    
    detected_emotion = classes[predicted_index]
    confidence_score = predictions[0][predicted_index] * 100
    
    print("\n================================")
    print("🎯 STANDALONE PREDICTION RESULT")
    print("================================")
    print(f"📁 File Name: {os.path.basename(TEST_FILE_PATH)}")
    print(f"🗣️ Detected Emotion: {detected_emotion.upper()}")
    print(f"📊 Confidence Match: {confidence_score:.2f}%")
    print("================================\n")


🎯 STANDALONE PREDICTION RESULT
📁 File Name: 1001_ITH_DIS_XX.wav
🗣️ Detected Emotion: SAD
📊 Confidence Match: 34.02%

